# Text classification - IMDB Movie reviews sentiment analysis
Loads IMDB reviews, preprocesses text, trains a Logistic Regression model, and predicts sentiment for user input.

In [1]:
import os
import re
import time
import warnings

import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [2]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):                          #tokenizing data
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)           # remove HTML tags
    text = re.sub(r'http\S+|www\S+', '', text)      # remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)         # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

In [3]:
df = pd.read_csv('reviews.csv')
print(f"{df.shape[0]} reviews loaded  ({df['label'].value_counts().to_dict()})")

print("Preprocessing text...")
start = time.time()
df['cleaned'] = df['text'].apply(preprocess_text)
df = df[df['cleaned'].str.strip().astype(bool)].reset_index(drop=True)
print(f"Done in {time.time() - start:.1f}s")

40000 reviews loaded  ({0: 20019, 1: 19981})
Preprocessing text...
Done in 39.6s


In [4]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X = vectorizer.fit_transform(df['cleaned'])
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

accuracy = accuracy_score(y_test, model.predict(X_test))
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.8825


In [5]:
user_input = input("Enter text to analyze: ")

cleaned = preprocess_text(user_input)
pred = model.predict(vectorizer.transform([cleaned]))[0]
proba = model.predict_proba(vectorizer.transform([cleaned]))[0]
label = 'Positive' if pred == 1 else 'Negative'
print(f"Sentiment: {label} ({max(proba):.2%})")

Sentiment: Negative (70.63%)
